# Datathon Round 1 MCQ Solver

Notebook nay giai 10 cau MCQ truc tiep tu cac file CSV trong `data/` bang `pandas`.

In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")


def locate_project_dir(start: Path | None = None) -> Path:
    start = Path.cwd().resolve() if start is None else start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data" / "orders.csv").exists():
            return candidate
    raise FileNotFoundError("Khong tim thay thu muc du an chua data/orders.csv")


PROJECT_DIR = locate_project_dir()
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "archive" / "sql_mcq"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

display(Markdown(f"**Project dir:** `{PROJECT_DIR}`"))
display(Markdown(f"**Data dir:** `{DATA_DIR}`"))
display(Markdown(f"**Output dir:** `{OUTPUT_DIR}`"))

**Project dir:** `C:\Users\Admin\Documents\VIN Datathon`

**Data dir:** `C:\Users\Admin\Documents\VIN Datathon\data`

**Output dir:** `C:\Users\Admin\Documents\VIN Datathon\archive\sql_mcq`

In [2]:
products = pd.read_csv(DATA_DIR / "products.csv")
customers = pd.read_csv(DATA_DIR / "customers.csv")
orders = pd.read_csv(DATA_DIR / "orders.csv", parse_dates=["order_date"])
order_items = pd.read_csv(
    DATA_DIR / "order_items.csv",
    usecols=["order_id", "product_id", "quantity", "unit_price", "discount_amount", "promo_id"],
)
payments = pd.read_csv(DATA_DIR / "payments.csv")
returns = pd.read_csv(DATA_DIR / "returns.csv", parse_dates=["return_date"])
web_traffic = pd.read_csv(DATA_DIR / "web_traffic.csv", parse_dates=["date"])
geography = pd.read_csv(DATA_DIR / "geography.csv")
sales = pd.read_csv(DATA_DIR / "sales.csv", parse_dates=["Date"])

table_summary = pd.DataFrame(
    [
        {"table": "products", "rows": len(products), "cols": products.shape[1]},
        {"table": "customers", "rows": len(customers), "cols": customers.shape[1]},
        {"table": "orders", "rows": len(orders), "cols": orders.shape[1]},
        {"table": "order_items", "rows": len(order_items), "cols": order_items.shape[1]},
        {"table": "payments", "rows": len(payments), "cols": payments.shape[1]},
        {"table": "returns", "rows": len(returns), "cols": returns.shape[1]},
        {"table": "web_traffic", "rows": len(web_traffic), "cols": web_traffic.shape[1]},
        {"table": "geography", "rows": len(geography), "cols": geography.shape[1]},
        {"table": "sales", "rows": len(sales), "cols": sales.shape[1]},
    ]
)

display(table_summary)

,table,rows,cols
0,products,2412,8
1,customers,121930,7
2,orders,646945,8
3,order_items,714669,6
4,payments,646945,4
5,returns,39939,7
6,web_traffic,3652,7
7,geography,39948,4
8,sales,3833,3


## Ghi chu ve cach tinh

- **Cau 1**: lay trung vi tren tat ca cac `inter-order gap` cua nhung khach hang co hon 1 don hang.
- **Cau 7**: `sales.csv` khong co cot `region`, nen doanh thu theo vung duoc tai dung tu `orders + order_items + geography`.
- **Cau 9**: giu dung cong thuc trong de: `so dong returns / so dong order_items`, khong dung `return_quantity`.

In [3]:
answers: list[dict] = []


def output_value(value):
    if isinstance(value, float):
        return round(value, 6)
    return value


def record_answer(mcq_no: int, question_short: str, computed_value, selected_option: str, selected_label: str, notes: str = ""):
    answers.append(
        {
            "mcq_no": mcq_no,
            "question_short": question_short,
            "computed_value": output_value(computed_value),
            "selected_option": selected_option,
            "selected_label": selected_label,
            "notes": notes,
        }
    )


def non_blank(series: pd.Series) -> pd.Series:
    return series.astype("string").str.strip().fillna("").ne("")

In [4]:
display(Markdown("## Q1 and Q2"))

orders_sorted = orders.sort_values(["customer_id", "order_date", "order_id"]).copy()
customer_order_counts = orders_sorted.groupby("customer_id")["order_id"].transform("count")
multi_orders = orders_sorted.loc[customer_order_counts.gt(1)].copy()
multi_orders["gap_days"] = multi_orders.groupby("customer_id")["order_date"].diff().dt.days
q1_gaps = multi_orders.loc[multi_orders["gap_days"].notna(), ["customer_id", "order_id", "gap_days"]]
q1_gap_median = float(q1_gaps["gap_days"].median())

display(Markdown(f"**Q1 median inter-order gap:** `{q1_gap_median:.1f}` days"))
display(q1_gaps["gap_days"].describe(percentiles=[0.25, 0.5, 0.75]).to_frame("gap_days"))
record_answer(
    1,
    "Median inter-order gap",
    q1_gap_median,
    "C",
    "180",
    "Gia tri tinh duoc la 144.0 ngay; dap an gan nhat trong bo lua chon la 180.",
)

products_margin = products.copy()
products_margin["gross_margin_ratio"] = (products_margin["price"] - products_margin["cogs"]) / products_margin["price"]
q2_table = (
    products_margin.groupby("segment", dropna=False)["gross_margin_ratio"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
q2_top_segment = q2_table.loc[0, "segment"]

display(Markdown(f"**Q2 top segment by avg gross margin ratio:** `{q2_top_segment}`"))
display(q2_table.head(10))
record_answer(2, "Top segment by gross margin ratio", q2_top_segment, "D", "Standard", "Match exact from grouped average margin ratio.")

## Q1 and Q2

**Q1 median inter-order gap:** `144.0` days

,gap_days
count,"556,699.000000"
mean,285.592509
std,389.691558
min,0.000000
25%,46.000000
50%,144.000000
75%,357.000000
max,"3,785.000000"


**Q2 top segment by avg gross margin ratio:** `Standard`

,segment,gross_margin_ratio
0,Standard,0.313442
1,Premium,0.285377
2,All-weather,0.284176
3,Activewear,0.265600
4,Performance,0.263650
5,Balanced,0.258038
6,Trendy,0.240758
7,Everyday,0.236343


In [5]:
display(Markdown("## Q3 and Q4"))

streetwear_returns = returns.merge(products[["product_id", "category"]], on="product_id", how="left")
q3_table = (
    streetwear_returns.loc[streetwear_returns["category"].eq("Streetwear"), "return_reason"]
    .value_counts(dropna=False)
    .rename_axis("return_reason")
    .reset_index(name="reason_count")
)
q3_reason = q3_table.loc[0, "return_reason"]

display(Markdown(f"**Q3 most common Streetwear return reason:** `{q3_reason}`"))
display(q3_table)
record_answer(3, "Most common Streetwear return reason", q3_reason, "B", "wrong_size", "Match exact from return counts after joining returns to products.")

q4_table = (
    web_traffic.groupby("traffic_source", dropna=False)["bounce_rate"]
    .mean()
    .sort_values()
    .reset_index(name="avg_bounce_rate")
)
q4_source = q4_table.loc[0, "traffic_source"]
q4_value = float(q4_table.loc[0, "avg_bounce_rate"])

display(Markdown(f"**Q4 lowest average bounce rate source:** `{q4_source}` ({q4_value:.6f})"))
display(q4_table)
record_answer(4, "Lowest avg bounce rate source", q4_value, "C", "email_campaign", f"Top source is {q4_source} with average bounce_rate {q4_value:.6f}.")

## Q3 and Q4

**Q3 most common Streetwear return reason:** `wrong_size`

,return_reason,reason_count
0,wrong_size,7626
1,defective,4330
2,not_as_described,3854
3,changed_mind,3830
4,late_delivery,2159


**Q4 lowest average bounce rate source:** `email_campaign` (0.004458)

,traffic_source,avg_bounce_rate
0,email_campaign,0.004458
1,social_media,0.004476
2,paid_search,0.004478
3,referral,0.004499
4,organic_search,0.004504
5,direct,0.004511


In [6]:
display(Markdown("## Q5 and Q6"))

promo_present = non_blank(order_items["promo_id"])
q5_pct = float(promo_present.mean() * 100)
q5_table = pd.DataFrame(
    {
        "metric": ["rows_with_promo_id", "all_order_item_rows", "pct_with_promo_id"],
        "value": [int(promo_present.sum()), int(len(order_items)), q5_pct],
    }
)

display(Markdown(f"**Q5 pct order_items with promo_id:** `{q5_pct:.6f}%`"))
display(q5_table)
record_answer(5, "Pct order_items with promo_id", q5_pct, "C", "39%", "Gia tri tinh duoc la 38.663493%, gan nhat voi 39%.")

orders_per_customer = orders.groupby("customer_id").size().rename("order_count")
q6_base = customers.loc[customers["age_group"].notna(), ["customer_id", "age_group"]].merge(
    orders_per_customer, on="customer_id", how="left"
)
q6_base["order_count"] = q6_base["order_count"].fillna(0)
q6_table = q6_base.groupby("age_group")["order_count"].agg(total_orders="sum", customers="count", avg_orders_per_customer="mean").sort_values(
    "avg_orders_per_customer", ascending=False
).reset_index()
q6_top_group = q6_table.loc[0, "age_group"]

display(Markdown(f"**Q6 highest avg orders per customer age group:** `{q6_top_group}`"))
display(q6_table)
record_answer(6, "Age group with highest avg orders/customer", q6_top_group, "A", "55+", "Computed as total orders divided by number of customers in each non-null age group.")

## Q5 and Q6

**Q5 pct order_items with promo_id:** `38.663493%`

,metric,value
0,rows_with_promo_id,"276,316.000000"
1,all_order_item_rows,"714,669.000000"
2,pct_with_promo_id,38.663493


**Q6 highest avg orders per customer age group:** `55+`

,age_group,total_orders,customers,avg_orders_per_customer
0,55+,"72,760.000000",13457,5.406851
1,45-54,"124,138.000000",23172,5.357241
2,35-44,"170,368.000000",31920,5.337343
3,25-34,"190,622.000000",36342,5.245226
4,18-24,"89,057.000000",17039,5.226656


In [7]:
display(Markdown("## Q7"))

q7_revenue = (
    order_items.merge(orders[["order_id", "zip"]], on="order_id", how="left")
    .merge(geography[["zip", "region"]], on="zip", how="left")
    .assign(
        gross_line_revenue=lambda df: df["quantity"] * df["unit_price"],
        net_line_revenue=lambda df: df["quantity"] * df["unit_price"] - df["discount_amount"],
    )
)

q7_table = (
    q7_revenue.groupby("region", dropna=False)[["gross_line_revenue", "net_line_revenue"]]
    .sum()
    .sort_values("gross_line_revenue", ascending=False)
    .reset_index()
)
q7_top_region = q7_table.loc[0, "region"]

display(Markdown("**Q7 reconstructed revenue by region**"))
display(q7_table)
record_answer(
    7,
    "Region with highest reconstructed revenue",
    q7_top_region,
    "C",
    "East",
    "sales.csv khong co region, nen doanh thu vung duoc tai dung tu orders + order_items + geography; East dung dau cho ca gross va net.",
)

## Q7

**Q7 reconstructed revenue by region**

,region,gross_line_revenue,net_line_revenue
0,East,"7,637,532,676.200000","7,291,150,819.120000"
1,Central,"4,941,908,471.680000","4,719,491,267.840000"
2,West,"3,851,035,437.650000","3,670,227,178.470000"


In [8]:
display(Markdown("## Q8, Q9 and Q10"))

q8_table = (
    orders.loc[orders["order_status"].eq("cancelled"), "payment_method"]
    .value_counts(dropna=False)
    .rename_axis("payment_method")
    .reset_index(name="cancelled_order_count")
)
q8_top_method = q8_table.loc[0, "payment_method"]
display(Markdown(f"**Q8 most used payment method for cancelled orders:** `{q8_top_method}`"))
display(q8_table)
record_answer(8, "Most used payment method for cancelled orders", q8_top_method, "A", "credit_card", "Match exact from cancelled order counts.")

returns_by_size = returns.merge(products[["product_id", "size"]], on="product_id", how="left")
items_by_size = order_items.merge(products[["product_id", "size"]], on="product_id", how="left")
valid_sizes = ["S", "M", "L", "XL"]

q9_num = returns_by_size.loc[returns_by_size["size"].isin(valid_sizes), "size"].value_counts().rename("return_rows")
q9_den = items_by_size.loc[items_by_size["size"].isin(valid_sizes), "size"].value_counts().rename("order_item_rows")
q9_table = pd.concat([q9_num, q9_den], axis=1).fillna(0)
q9_table["return_rate"] = q9_table["return_rows"] / q9_table["order_item_rows"]
q9_table = q9_table.sort_values("return_rate", ascending=False).reset_index(names="size")
q9_top_size = q9_table.loc[0, "size"]
q9_top_rate = float(q9_table.loc[0, "return_rate"])

display(Markdown(f"**Q9 highest return-rate size:** `{q9_top_size}` ({q9_top_rate:.6f})"))
display(q9_table)
record_answer(9, "Highest return rate by size", q9_top_rate, "A", "S", f"Top size is {q9_top_size} with return-rate {q9_top_rate:.6f} using row-count formula from the brief.")

q10_table = payments.groupby("installments", dropna=False)["payment_value"].mean().sort_values(ascending=False).reset_index(name="avg_payment_value")
q10_top_installments = int(q10_table.loc[0, "installments"])
q10_top_value = float(q10_table.loc[0, "avg_payment_value"])

display(Markdown(f"**Q10 installment plan with highest avg payment_value:** `{q10_top_installments}` ({q10_top_value:.6f})"))
display(q10_table)
record_answer(10, "Installment plan with highest avg payment", q10_top_value, "C", "6", f"Highest average payment_value belongs to installments={q10_top_installments}.")

## Q8, Q9 and Q10

**Q8 most used payment method for cancelled orders:** `credit_card`

,payment_method,cancelled_order_count
0,credit_card,28452
1,cod,15468
2,paypal,7817
3,apple_pay,5190
4,bank_transfer,2535


**Q9 highest return-rate size:** `S` (0.056515)

,size,return_rows,order_item_rows,return_rate
0,S,9723,172042,0.056515
1,L,9741,173174,0.056250
2,M,9820,176428,0.055660
3,XL,10655,193025,0.055200


**Q10 installment plan with highest avg payment_value:** `6` (24446.654403)

,installments,avg_payment_value
0,6,"24,446.654403"
1,3,"24,399.635486"
2,12,"24,245.772694"
3,1,"24,113.274166"
4,2,708.473729


In [9]:
answers_df = pd.DataFrame(answers).sort_values("mcq_no").reset_index(drop=True)
display(Markdown("## Final answer table"))
display(answers_df)

answer_key_df = answers_df[["mcq_no", "selected_option", "selected_label", "computed_value", "notes"]].copy()

answers_path = OUTPUT_DIR / "mcq_answers.csv"
answer_key_path = OUTPUT_DIR / "answer_key.csv"

answers_df.to_csv(answers_path, index=False)
answer_key_df.to_csv(answer_key_path, index=False)

display(Markdown(f"Saved `{answers_path}`"))
display(Markdown(f"Saved `{answer_key_path}`"))

## Final answer table

,mcq_no,question_short,computed_value,selected_option,selected_label,notes
0,1,Median inter-order gap,144.000000,C,180,Gia tri tinh duoc la 144.0 ngay; dap an gan nh...
1,2,Top segment by gross margin ratio,Standard,D,Standard,Match exact from grouped average margin ratio.
2,3,Most common Streetwear return reason,wrong_size,B,wrong_size,Match exact from return counts after joining r...
3,4,Lowest avg bounce rate source,0.004458,C,email_campaign,Top source is email_campaign with average boun...
4,5,Pct order_items with promo_id,38.663493,C,39%,"Gia tri tinh duoc la 38.663493%, gan nhat voi ..."
5,6,Age group with highest avg orders/customer,55+,A,55+,Computed as total orders divided by number of ...
6,7,Region with highest reconstructed revenue,East,C,East,"sales.csv khong co region, nen doanh thu vung ..."
7,8,Most used payment method for cancelled orders,credit_card,A,credit_card,Match exact from cancelled order counts.
8,9,Highest return rate by size,0.056515,A,S,Top size is S with return-rate 0.056515 using ...
9,10,Installment plan with highest avg payment,"24,446.654403",C,6,Highest average payment_value belongs to insta...


Saved `C:\Users\Admin\Documents\VIN Datathon\archive\sql_mcq\mcq_answers.csv`

Saved `C:\Users\Admin\Documents\VIN Datathon\archive\sql_mcq\answer_key.csv`